# Visual smoke experiment: real, generated, and standard augmented images

This notebook uses the existing generation, augmentation, training, and evaluation blocks directly, so their messages and progress bars appear here.

You will see:
- Real and generated examples of the **same species** as soon as generation finishes.
- Original photos, baseline preprocessing, and the **exact fixed augmented views** used in training.
- Live loss, validation accuracy, and validation macro F1 plots after each validation checkpoint.
- Test metrics and up to five wrong predictions, with image previews and links, for every setup.

Each new notebook run creates a fresh image pool with new generation seeds, recorded in its settings. All three setups use the same real subset and optimizer-step budget; all training finishes before test evaluation.

In [1]:
import gc
import importlib
import hashlib
import json
import sys
import uuid
from pathlib import Path

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src" / "load_data.py").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd
import torch
from datasets import load_from_disk
from IPython import get_ipython
from IPython.display import Markdown, display
from tqdm.auto import tqdm

if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")

# Refresh cached project modules before binding functions in this notebook.
# Dependencies go first so loaders, dataset classes and previews stay consistent.
importlib.invalidate_caches()
for module_name in (
    "load_data", "synthetic_data", "preprocessing", "classifier", "leakage",
    "prompts", "generate", "prepare_synthetic", "train_classifier",
    "evaluate_classifier", "smoke_experiment", "smoke_visualization",
):
    cached_module = sys.modules.get(module_name)
    if cached_module is not None:
        importlib.reload(cached_module)

from classifier import class_names_from_manifest, resolve_device
from load_data import (
    DATA_DIR, PROCESSED_DATA_DIR, canonical_sha256, load_manifest, manifest_path, sha256_matches,
)
from smoke_experiment import SETUPS, write_report
from smoke_visualization import (
    SETUP_NAMES, LiveTrainingPlot, plot_history, save_figure, show_augmentations,
    show_generated_pairs, show_test_comparison, show_wrong_predictions,
)
from synthetic_data import load_synthetic_manifest
from preprocessing import select_augmentation_source

plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
print(f"Project: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")

Project: c:\Users\КАРИНА\PythonProjects\cv-data-augmentation-experiment
Python: c:\Users\КАРИНА\PythonProjects\cv-data-augmentation-experiment\.venv\Scripts\python.exe


## Settings

The default is **200 generated images** (one per species), 10 real training images per species, and **20 optimizer steps per setup**, validating every 5 steps so that the learning curves contain four points. These are smoke-test scores.

Set `MAX_STEPS = 2` and `EVAL_EVERY = 2` to match the original script's shortest run; that produces one point per training chart. Use 1,000 steps / validation every 100 for the previous experiment budget.

Running the notebook from the settings cell creates a new run directory, pool and generation seeds. Rerunning an individual stage within that run resumes its pool or redisplays its completed results. Set a fixed `RUN_DIR` explicitly to resume the same run after restarting the kernel; incomplete preparation/training stages require a new `RUN_DIR`.

In [2]:
RUN_DIR = PROJECT_ROOT / "outputs" / f"notebook_smoke_{uuid.uuid4().hex[:8]}"

SEED = 0
K_SHOT = 10
PER_CLASS = 1
AUGMENT_PER_CLASS = 1  # One fixed augmented view per species: 200 added views.
TEMPLATE = "wildlife"
GENERATION_STEPS = 25
GENERATION_BATCH_SIZE = 1
GENERATION_SEED_OFFSET = None  # Automatically choose and record new seeds for each fresh run.
ALLOW_CPU_GENERATION = False
REVIEW_CSV = None  # Optional review CSV for this pool; otherwise only integrity/leakage filtering.

MAX_STEPS = 20
EVAL_EVERY = 5
BATCH_SIZE = 64
NUM_WORKERS = 0
DEVICE = "auto"
AMP = torch.cuda.is_available()
PREVIEW_SPECIES = 6

In [4]:
device = resolve_device(DEVICE)
if AMP and device.type != "cuda":
    raise ValueError("AMP requires CUDA; set AMP = False for CPU.")
if not PROCESSED_DATA_DIR.exists():
    raise FileNotFoundError("Run python src/load_data.py first.")
if not torch.cuda.is_available() and not ALLOW_CPU_GENERATION:
    raise ValueError("Generation requires CUDA; explicitly enable ALLOW_CPU_GENERATION for slow CPU diffusion.")
if K_SHOT not in (5, 10, 20) or SEED not in (0, 1, 2):
    raise ValueError("Use an existing 5/10/20-shot subset with seed 0, 1 or 2.")
if min(PER_CLASS, AUGMENT_PER_CLASS, GENERATION_STEPS, GENERATION_BATCH_SIZE, MAX_STEPS, EVAL_EVERY, BATCH_SIZE, PREVIEW_SPECIES) <= 0:
    raise ValueError("Image counts, step budgets and batch sizes must be positive.")
if AUGMENT_PER_CLASS > K_SHOT:
    raise ValueError("AUGMENT_PER_CLASS cannot exceed K_SHOT.")
if NUM_WORKERS < 0:
    raise ValueError("NUM_WORKERS must be nonnegative.")

RUN_DIR = RUN_DIR.resolve()
config_path = RUN_DIR / "smoke_config.json"
previous_notebook = json.loads(config_path.read_text(encoding="utf-8")) if config_path.exists() else None
POOL = previous_notebook["pool"] if previous_notebook else f"smoke_notebook_{uuid.uuid4().hex[:12]}"
if GENERATION_SEED_OFFSET is None and previous_notebook is not None:
    GENERATION_SEED_OFFSET = previous_notebook.get("generation_seed_offset")
if GENERATION_SEED_OFFSET is None:
    GENERATION_SEED_OFFSET = 1_000_000 + int(uuid.uuid4().hex[:8], 16)
if not isinstance(GENERATION_SEED_OFFSET, int) or not 0 <= GENERATION_SEED_OFFSET < 2 ** 63 - 200_000:
    raise ValueError("GENERATION_SEED_OFFSET must be a nonnegative integer below 2**63 - 200000.")
POOL_DIR = DATA_DIR / "synthetic" / POOL
PREPARED_DIR = RUN_DIR / "prepared_synthetic"
SYNTHETIC_MANIFEST = PREPARED_DIR / "manifest.csv"
FIGURES = RUN_DIR / "figures"
if REVIEW_CSV is not None:
    REVIEW_CSV = Path(REVIEW_CSV).resolve()

settings = dict(
    interface="notebook", output=str(RUN_DIR), pool=POOL, seed=SEED, k_shot=K_SHOT,
    per_class=PER_CLASS, template=TEMPLATE, generation_steps=GENERATION_STEPS,
    augment_per_class=AUGMENT_PER_CLASS,
    generation_batch_size=GENERATION_BATCH_SIZE, allow_cpu_generation=ALLOW_CPU_GENERATION,
    generation_seed_offset=GENERATION_SEED_OFFSET,
    review=None if REVIEW_CSV is None else str(REVIEW_CSV),
    review_sha256=None if REVIEW_CSV is None else canonical_sha256(REVIEW_CSV),
    max_steps=MAX_STEPS, eval_every=EVAL_EVERY, batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS, device=str(device), amp=AMP, setups=list(SETUPS),
    checkpoint_selection="val_macro_f1", evaluation_split="test",
    synthetic_manifest=str(SYNTHETIC_MANIFEST),
    real_manifest_sha256={
        split: canonical_sha256(manifest_path(split, K_SHOT, SEED) if split == "train" else manifest_path(split))
        for split in ("train", "val", "test")
    },
)
if previous_notebook is not None and previous_notebook != settings:
    raise ValueError("RUN_DIR contains different settings. Choose a new RUN_DIR.")
if previous_notebook is None:
    if RUN_DIR.exists():
        raise FileExistsError("RUN_DIR already exists without matching notebook settings. Choose a new directory.")
    RUN_DIR.mkdir(parents=True)
    config_path.write_text(json.dumps(settings, indent=2), encoding="utf-8")
FIGURES.mkdir(exist_ok=True)

train_manifest = load_manifest("train", K_SHOT, SEED)
names = class_names_from_manifest(train_manifest)
raw_train = load_from_disk(str(PROCESSED_DATA_DIR))["train"]
row_of = {image_id: index for index, image_id in enumerate(raw_train["image_id"])}
real_train = raw_train.select([row_of[image_id] for image_id in train_manifest.image_id])

overall = tqdm(total=10, desc="Notebook workflow", unit="stage")
completed = set()
def mark_done(stage):
    completed.add(stage)
    overall.n = len(completed)
    overall.set_postfix_str(stage)
    overall.refresh()

def release_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"Device: {device}; AMP: {AMP}")
print(f"Generation pool: {POOL}; seed offset: {GENERATION_SEED_OFFSET}")
print(f"Real subset: {len(real_train):,} images; generation target: {len(names) * PER_CLASS:,} images.")
print(f"Each classifier: {MAX_STEPS} optimizer steps; validation every {EVAL_EVERY} steps.")
print(f"Results: {RUN_DIR}")

Notebook workflow:   0%|          | 0/10 [00:00<?, ?stage/s]

Device: cuda; AMP: True
Generation pool: smoke_notebook_0ff61d18a452; seed offset: 3675888794
Real subset: 2,000 images; generation target: 200 images.
Each classifier: 20 optimizer steps; validation every 5 steps.
Results: C:\Users\КАРИНА\PythonProjects\cv-data-augmentation-experiment\outputs\notebook_smoke_0ebdc232


## 1. Generate images and compare species

A fresh run generates all 200 images from scratch using its new seeds. Rerunning this cell within the same run resumes generation with the recorded seeds. The gallery uses only real **training** images and raw generated images, before integrity/leakage filtering.

In [ ]:
from generate import generate, report_cost, write_manifest

metadata_path = POOL_DIR / "metadata.csv"
before_ids = ({image_id for image_id in pd.read_csv(metadata_path).image_id
               if (DATA_DIR / f"{image_id}.jpg").is_file()} if metadata_path.exists() else set())
generate(
    POOL, names, PER_CLASS, TEMPLATE, batch_size=GENERATION_BATCH_SIZE,
    steps=GENERATION_STEPS, allow_cpu=ALLOW_CPU_GENERATION, seed_offset=GENERATION_SEED_OFFSET,
)
metadata, generated_manifest_path = write_manifest(POOL)
new_count = len(set(metadata.image_id) - before_ids)
print(f"{len(metadata):,} images generated in the pool ({new_count:,} new; {len(metadata) - new_count:,} reused).")
report_cost(metadata)
release_gpu_memory()

fig = show_generated_pairs(
    real_train, metadata, names, n=PREVIEW_SPECIES, seed=SEED,
    save_path=FIGURES / "real_vs_generated.png",
)
display(fig)
plt.close(fig)
mark_done("Generation and species gallery")

## 2. Inspect standard augmentation

The smoke test adds **200 fixed crop/flip views** by default: one selected real training image per species (`AUGMENT_PER_CLASS = 1`). All 2,000 original real images stay in training, giving 2,200 entries. Source photos are selected reproducibly by image ID within each species. The preview uses those same source photos and augmentation seeds. The middle column shows baseline preprocessing; the right column is the added training view. Views are created lazily, without extra JPEG files.

In [ ]:
augmented_count = len(select_augmentation_source(real_train, AUGMENT_PER_CLASS))
print(f"{augmented_count:,} fixed augmented views configured; "
      f"{len(real_train) + augmented_count:,} total entries in the standard augmentation setup.")
fig = show_augmentations(
    real_train, names, n=PREVIEW_SPECIES, seed=SEED,
    save_path=FIGURES / "real_vs_augmented.png", augment_per_class=AUGMENT_PER_CLASS,
)
display(fig)
plt.close(fig)
mark_done("Augmentation gallery")

## 3. Prepare the generated pool

Reuse the existing integrity, hash, and validation/test leakage checks. Validation/test images are used here only for duplicate detection, without classifier metrics. Species review is applied when `REVIEW_CSV` is supplied.

In [ ]:
from prepare_synthetic import main as prepare_synthetic

if PREPARED_DIR.exists():
    if not SYNTHETIC_MANIFEST.is_file() or not (PREPARED_DIR / "leakage_report.json").is_file():
        raise FileExistsError("Incomplete preparation stage; choose a new RUN_DIR.")
    leakage = json.loads((PREPARED_DIR / "leakage_report.json").read_text(encoding="utf-8"))
    if leakage["metadata_sha256"] != hashlib.sha256(metadata_path.read_bytes()).hexdigest():
        raise ValueError("Generation metadata changed after preparation; choose a new RUN_DIR.")
    print("Reusing completed synthetic preparation.")
else:
    arguments = [
        "--pool", str(POOL_DIR), "--output", str(PREPARED_DIR), "--device", str(device),
        "--batch-size", str(BATCH_SIZE), "--expected-per-class", str(PER_CLASS),
        "--per-class", str(PER_CLASS),
    ]
    if REVIEW_CSV is not None:
        arguments += ["--review", str(REVIEW_CSV)]
    prepare_synthetic(arguments)

synthetic = load_synthetic_manifest(SYNTHETIC_MANIFEST, names)
print(f"{len(synthetic):,} generated images accepted for training "
      f"across {synthetic.label.nunique()} species.")
display(pd.DataFrame([
    {"Setup": "Real only", "Real": len(real_train), "Generated": 0, "Augmented views": 0},
    {"Setup": "Real + generated", "Real": len(real_train), "Generated": len(synthetic), "Augmented views": 0},
    {"Setup": "Real + standard augmentation", "Real": len(real_train), "Generated": 0, "Augmented views": augmented_count},
]))
release_gpu_memory()
mark_done("Synthetic preparation")

## 4. Train all three setups with live curves

The figures update **after each validation checkpoint**; optimizer and validation bars continue between updates. Training loss is the mean since the previous validation. The star marks the checkpoint with the best validation macro F1.

The same architecture, real subset, seed, optimizer defaults and step budget are used in every setup. No test scores are available during training. Completed matching runs are redisplayed without retraining.

In [ ]:
from train_classifier import main as train_classifier

common = [
    "--seed", str(SEED), "--k-shot", str(K_SHOT), "--max-steps", str(MAX_STEPS),
    "--eval-every", str(EVAL_EVERY), "--log-every", "1", "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS), "--device", str(device),
]
if AMP:
    common.append("--amp")

for setup in tqdm(SETUPS, desc="Training setups", unit="setup"):
    print(f"\nTraining: {SETUP_NAMES[setup]}")
    train_dir = RUN_DIR / setup / "train"
    figure_path = FIGURES / f"{setup}_training.png"
    if train_dir.exists():
        required = ("config.json", "history.csv", "best.pt", "runtime.json", "val_metrics.json")
        if not all((train_dir / name).is_file() for name in required):
            raise FileExistsError(f"Incomplete training stage: {train_dir}. Choose a new RUN_DIR.")
        config = json.loads((train_dir / "config.json").read_text(encoding="utf-8"))
        expected = dict(seed=SEED, k_shot=K_SHOT, max_steps=MAX_STEPS, eval_every=EVAL_EVERY,
                        batch_size=BATCH_SIZE, amp=AMP, augment=setup == "real_standard_augmented",
                        augment_per_class=AUGMENT_PER_CLASS if setup == "real_standard_augmented" else None,
                        synthetic_manifest=str(SYNTHETIC_MANIFEST) if setup == "real_generated" else None)
        if any(config.get(key) != value for key, value in expected.items()):
            raise ValueError(f"Saved training settings differ for {setup}; choose a new RUN_DIR.")
        print("Using completed training artifacts.")
        fig = plot_history(pd.read_csv(train_dir / "history.csv"), SETUP_NAMES[setup])
        save_figure(fig, figure_path)
        display(fig)
        plt.close(fig)
    else:
        extra = (["--synthetic-manifest", str(SYNTHETIC_MANIFEST)] if setup == "real_generated" else
                 ["--augment", "--augment-per-class", str(AUGMENT_PER_CLASS)]
                 if setup == "real_standard_augmented" else [])
        live = LiveTrainingPlot(SETUP_NAMES[setup], save_path=figure_path)
        try:
            train_classifier(common + extra + ["--output", str(train_dir)], on_validation=live.update)
        finally:
            live.finish()
            release_gpu_memory()
    runtime = json.loads((train_dir / "runtime.json").read_text(encoding="utf-8"))
    print(f"{SETUP_NAMES[setup]} trained: {runtime['optimizer_steps']} steps; "
          f"best validation macro F1 = {runtime['best_val_macro_f1']:.4f} at step {runtime['best_step']}.")
    mark_done(f"Trained {setup}")

## 5. Predict the full real test set

All three models must be trained first. Each evaluation loads its validation-selected checkpoint, verifies the test manifest, and computes accuracy, macro F1, loss, and per-class precision/recall/F1. Test progress covers the complete split, including the last partial batch.

In [ ]:
from evaluate_classifier import main as evaluate_classifier

if not all((RUN_DIR / setup / "train" / "runtime.json").is_file() for setup in SETUPS):
    raise RuntimeError("Finish training all three setups before evaluating test.")

for setup in tqdm(SETUPS, desc="Test setups", unit="setup"):
    print(f"\nPredicting test: {SETUP_NAMES[setup]}")
    checkpoint = RUN_DIR / setup / "train" / "best.pt"
    test_dir = RUN_DIR / setup / "test"
    if test_dir.exists():
        required = ("test_metrics.json", "test_per_class.json", "test_predictions.csv", "evaluation_config.json")
        if not all((test_dir / name).is_file() for name in required):
            raise FileExistsError(f"Incomplete test evaluation: {test_dir}. Choose a new RUN_DIR.")
        evaluation_config = json.loads((test_dir / "evaluation_config.json").read_text(encoding="utf-8"))
        if (evaluation_config["checkpoint_sha256"] != hashlib.sha256(checkpoint.read_bytes()).hexdigest()
                or not sha256_matches(manifest_path("test"), evaluation_config["manifest_sha256"])):
            raise ValueError("Checkpoint or test manifest changed; choose a new RUN_DIR.")
        print("Using completed test predictions.")
    else:
        evaluate_classifier([
            str(checkpoint), "--split", "test", "--output", str(test_dir), "--device", str(device),
            "--batch-size", str(BATCH_SIZE), "--num-workers", str(NUM_WORKERS),
        ])
    metrics = json.loads((test_dir / "test_metrics.json").read_text(encoding="utf-8"))
    print(f"{metrics['images']:,} test predictions: accuracy={metrics['accuracy']:.4f}, "
          f"macro F1={metrics['macro_f1']:.4f}.")
    release_gpu_memory()
    mark_done(f"Tested {setup}")

## 6. Compare test results and inspect mistakes

The table and bar charts use the project's saved test metrics. Every setup also gets the first five wrong predictions in test-manifest order, with true/predicted species, local image links, and an image gallery. If there are fewer than five mistakes, all available mistakes are shown.

In [ ]:
write_report(RUN_DIR)
summary = pd.read_csv(RUN_DIR / "metrics_summary.csv")
display(summary[[
    "setup", "real_train_images", "synthetic_train_images", "augmented_train_images",
    "images", "loss", "accuracy", "macro_f1",
]])
fig = show_test_comparison(summary, save_path=FIGURES / "test_comparison.png")
display(fig)
plt.close(fig)

display(Markdown(
    f"[Open full report]({(RUN_DIR / 'summary.md').as_uri()}) | "
    f"[Metric summary CSV]({(RUN_DIR / 'metrics_summary.csv').as_uri()})"
))
for setup in SETUPS:
    fig = show_wrong_predictions(RUN_DIR, setup, save_path=FIGURES / f"{setup}_mistakes.png")
    if fig is not None:
        display(fig)
        plt.close(fig)
    wrong = pd.read_csv(RUN_DIR / setup / "test" / "wrong_predictions.csv")
    for row in wrong.itertuples(index=False):
        display(Markdown(
            f"[{row.image_id}]({row.image_url}) — true: **{row.class_name}**; "
            f"predicted: **{row.predicted_class_name}**"
        ))
    # Show per-class precision/recall/F1 for the species pictured in the galleries.
    per_class = json.loads((RUN_DIR / setup / "test" / "test_per_class.json").read_text(encoding="utf-8"))
    pictured = sorted(set(wrong.class_name)) if len(wrong) else names[:PREVIEW_SPECIES]
    display(Markdown(f"**{SETUP_NAMES[setup]}: per-class metrics for the pictured true species**"))
    display(pd.DataFrame({name: per_class[name] for name in pictured}).T[["precision", "recall", "f1-score", "support"]])

mark_done("Report and error galleries")
overall.close()
print(f"All three setups complete. Figures, metrics, checkpoints and image links are saved in {RUN_DIR}.")

All figures are saved as PNG files under `RUN_DIR/figures/`; model and evaluation artifacts keep the same format as the command-line workflow. Rerun a completed cell to redisplay its results. Change `RUN_DIR` when changing experimental settings or recovering an incomplete stage.

For a larger experiment, freeze settings before running all setups. Fresh generated pools still need species-fidelity review for research-quality conclusions.